# Step 3: run the open models on Colab

Runtime → Change runtime type → **T4 GPU**. Run the cells top to bottom. Each run appends to `outputs/raw/<model>/`, so if Colab disconnects, run the same cell again and it resumes.

Order: do the dry run first and check the printed outputs before the full runs.

In [ ]:
!git clone https://github.com/shuaige387-a11y/LLM_Wug-Test_on_Dutch_Diminutives.git repo
%cd repo
!pip install -q -r requirements-gpu.txt

Some models (Gemma) ask you to accept their license on Hugging Face first. Accept it on the model page while logged in, then paste a read token here.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## Dry run: 3 words, one small model
Check that the scores look sane and the generated answers are short Dutch words.

In [ ]:
!python scripts/03_run_hf.py --model eurollm-1.7b --method B --limit 3
!python scripts/03_run_hf.py --model eurollm-1.7b-instruct --method A --limit 3

In [ ]:
import pandas as pd
s = pd.read_json('outputs/raw/eurollm-1.7b/scoring.jsonl', lines=True)
display(s[s.context=='frame'][['stem','candidate_form','is_correct','logprob_total','n_tokens','tokens']])
g = pd.read_json('outputs/raw/eurollm-1.7b-instruct/generation.jsonl', lines=True)
display(g[g.decoding=='greedy'][['stem','template','output']])

Delete the dry-run files before the full run, so the full run starts clean:

In [ ]:
!rm -rf outputs/raw/eurollm-1.7b outputs/raw/eurollm-1.7b-instruct

## Full runs
One model per cell. `--words core` = 15 core nonce words + 20 real words; add `--words all` later for the 51 reserve words. Restart the runtime between big models if memory runs out. Each model's download cache is deleted after its last run, so the 112 GB Colab disk does not fill up. Check free space any time with `!df -h /content`.

In [ ]:
RUNS = [('eurollm-1.7b','B'), ('eurollm-1.7b','A'), ('eurollm-1.7b-instruct','A'), ('fietje-2','B'), ('fietje-2','A')]
for m, meth in RUNS:
    last = all(m2 != m for m2, _ in RUNS[RUNS.index((m, meth)) + 1:])
    flag = '--clear-cache' if last else ''
    !python scripts/03_run_hf.py --model {m} --method {meth} {flag}

In [ ]:
RUNS = [('gemma4-e2b','B'), ('gemma4-e2b-it','A'), ('gemma4-e4b','B'), ('gemma4-e4b-it','A')]
for m, meth in RUNS:
    last = all(m2 != m for m2, _ in RUNS[RUNS.index((m, meth)) + 1:])
    flag = '--clear-cache' if last else ''
    !python scripts/03_run_hf.py --model {m} --method {meth} {flag}

In [ ]:
RUNS = [('eurollm-9b','B'), ('eurollm-9b-instruct','A'), ('gemma4-12b','B'), ('gemma4-12b-it','A')]
for m, meth in RUNS:
    last = all(m2 != m for m2, _ in RUNS[RUNS.index((m, meth)) + 1:])
    flag = '--clear-cache' if last else ''
    !python scripts/03_run_hf.py --model {m} --method {meth} {flag}

## Save the outputs
Download the zip and upload it to the project, or commit it from here.

In [ ]:
!zip -qr outputs_raw.zip outputs/raw
from google.colab import files
files.download('outputs_raw.zip')